<a href="https://colab.research.google.com/github/jeevanshrestha/GenAi/blob/main/RAG_With_Knowledge_graph(Neo4j).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# langchain-core

contains simple, core abstractions that have emerged as a standard, as well as LangChain Expression Language as a way to compose these components together. This package is now at version 0.1 and all breaking changes will be accompanied by a minor version bump.

# langchain-community
contains all third party integrations. We will work with partners on splitting key integrations out into standalone packages over the next month.

# langchain
contains higher-level and use-case specific chains, agents, and retrieval algorithms that are at the core of your application's cognitive architecture. We are targeting a launch of a stable 0.1 release for langchain in early January.#

In [1]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.0 MB/s eta 0:00:00


In [2]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [5]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('genai_course')

In [6]:
from typing import Tuple, List, Optional

In [7]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [8]:
from langchain_core.runnables import ConfigurableField

In [9]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase


In [10]:
import os

In [11]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [12]:
from langchain_community.vectorstores import Neo4jVector

In [13]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('genai_course')

In [37]:
NEO4J_URI="neo4j+s://5dc50d95.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD=userdata.get('NEO4J_PASSWORD')

In [75]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [76]:
from langchain_community.graphs import Neo4jGraph

In [77]:
graph = Neo4jGraph()

In [78]:
from langchain.document_loaders import WikipediaLoader
raw_documents = WikipediaLoader(query="Queen Elizabeth II").load()

In [79]:
len(raw_documents)

25

In [80]:
raw_documents[:3]

[Document(metadata={'title': 'Elizabeth II', 'summary': "Elizabeth II (Elizabeth Alexandra Mary; 21 April 1926 – 8 September 2022) was Queen of the United Kingdom and other Commonwealth realms from 6 February 1952 until her death in 2022. She had been queen regnant of 32 sovereign states during her lifetime and was the monarch of 15 realms at her death. Her reign of 70 years and 214 days is the longest of any British monarch, the second-longest of any sovereign state, and the longest of any queen regnant in history.\nElizabeth was born in Mayfair, London, during the reign of her paternal grandfather, King George V. She was the first child of the Duke and Duchess of York (later King George VI and Queen Elizabeth The Queen Mother). Her father acceded to the throne in 1936 upon the abdication of his brother Edward VIII, making the ten-year-old Princess Elizabeth the heir presumptive. She was educated privately at home and began to undertake public duties during the Second World War, servi

In [81]:
from langchain.text_splitter import TokenTextSplitter
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])

In [82]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-0125")

In [83]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(llm=llm)

/usr/local/lib/python3.11/dist-packages/langchain_openai/chat_models/base.py:1772: UserWarning:

Cannot use method='json_schema' with model gpt-3.5-turbo-0125 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.



In [84]:
graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [85]:
graph_documents

[GraphDocument(nodes=[Node(id='Elizabeth Ii', type='Person', properties={}), Node(id='United Kingdom', type='Country', properties={}), Node(id='Commonwealth Realms', type='Country', properties={}), Node(id='King George V', type='Person', properties={}), Node(id='Duke Of York', type='Person', properties={}), Node(id='Queen Elizabeth The Queen Mother', type='Person', properties={}), Node(id='King George Vi', type='Person', properties={}), Node(id='Edward Viii', type='Person', properties={}), Node(id='Auxiliary Territorial Service', type='Organization', properties={}), Node(id='Philip Mountbatten', type='Person', properties={}), Node(id='Charles', type='Person', properties={}), Node(id='Anne', type='Person', properties={}), Node(id='Andrew', type='Person', properties={}), Node(id='Edward', type='Person', properties={}), Node(id='Canada', type='Country', properties={}), Node(id='Australia', type='Country', properties={}), Node(id='New Zealand', type='Country', properties={}), Node(id='Sout

In [86]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [87]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

In [88]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [89]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [90]:
def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    display(widget)
    return widget

In [91]:
showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

GraphWidget(layout=Layout(height='800px', width='100%'))

In [92]:
from typing import Tuple, List, Optional

In [93]:
from langchain_community.vectorstores import Neo4jVector

In [94]:
from langchain_openai import OpenAIEmbeddings
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

In [95]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

[]

In [101]:
from langchain_core.pydantic_v1 import BaseModel, Field
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )


In [102]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [103]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [104]:
entity_chain = prompt | llm.with_structured_output(Entities)

/usr/local/lib/python3.11/dist-packages/langchain_openai/chat_models/base.py:1759: UserWarning:

Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".



In [105]:
entity_chain.invoke({"question": "Where was Amelia Earhart born?"}).names

['Amelia Earhart']

In [106]:
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

In [107]:
def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


In [108]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [110]:
print(structured_retriever("Who is Elizabeth II?"))

Elizabeth Ii - REIGNED -> United Kingdom
Elizabeth Ii - REIGNED -> Commonwealth Realms
Elizabeth Ii - REIGNED -> Canada
Elizabeth Ii - REIGNED -> Australia
Elizabeth Ii - REIGNED -> New Zealand
Elizabeth Ii - REIGNED -> South Africa
Elizabeth Ii - REIGNED -> Pakistan
Elizabeth Ii - REIGNED -> Ceylon
Elizabeth Ii - REIGNED -> Northern Ireland
Elizabeth Ii - FAMILY -> King George V
Elizabeth Ii - FAMILY -> Duke Of York
Elizabeth Ii - FAMILY -> Queen Elizabeth The Queen Mother
Elizabeth Ii - FAMILY -> King George Vi
Elizabeth Ii - FAMILY -> Edward Viii
Elizabeth Ii - SERVED -> Auxiliary Territorial Service
Elizabeth Ii - MARRIED -> Philip Mountbatten
Elizabeth Ii - VISITED -> Europe
Elizabeth Ii - VISITED -> China
Elizabeth Ii - VISITED -> Russia
Elizabeth Ii - VISITED -> Republic Of Ireland
Elizabeth Ii - MET -> Pope
Elizabeth Ii - MET -> Us President
Elizabeth Ii - DIED -> Balmoral Castle
Elizabeth Ii - DIED -> Scotland
Elizabeth Ii - SUCCEEDED -> Charles Iii
Elizabeth Ii - INVOLVED_IN 

In [111]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

In [112]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [113]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [114]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [115]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [116]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

In [118]:
prompt = ChatPromptTemplate.from_template(template)

In [119]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [120]:
chain.invoke({"question": "Which house did Elizabeth II belong to?"})

Search query: Which house did Elizabeth II belong to?


'House of Windsor'

In [122]:
chain.invoke(
    {
        "question": "When was she born?",
        "chat_history": [("Which house did Elizabeth II belong to?", "House Of Tudor")],
    }
)

Search query: When was Elizabeth II born?


'Elizabeth II was born on April 21, 1926.'